<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/glasslego/ml-deep-learning-study/blob/main/src/deep_learning_이론/05_universal_approximation_theorem.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

# Chapter 5: 근사 이론 (Universal Approximation Theorem)

이 노트북에서는 신경망의 이론적 기초인 **범용 근사 정리(Universal Approximation Theorem)**를 학습합니다.

## 🎯 학습 목표
- Universal Approximation Theorem의 개념과 의미 이해
- 정리의 수학적 증명 과정 학습
- 정리의 한계와 실제 적용 시 고려사항 파악
- 다양한 활성화 함수에서의 근사 능력 실험

## 📚 주요 내용
1. **Universal Approximation Theorem 개념**
2. **수학적 증명**
3. **정리의 한계와 실제 적용**
4. **실험을 통한 검증**

## 1. 환경 설정 및 라이브러리 임포트

In [ ]:
# 기본 라이브러리 임포트
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Google Colab 환경 체크
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # 나눔고딕 폰트 설치
    !sudo apt-get install -y fonts-nanum > /dev/null 2>&1
    !sudo fc-cache -fv > /dev/null 2>&1
    !rm ~/.cache/matplotlib -rf
    
    # 폰트 설정
    plt.rc('font', family='NanumGothic')
else:
    # 로컬 환경 폰트 설정
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지

# TensorFlow/Keras 임포트
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import models, layers, optimizers
    print(f"TensorFlow 버전: {tf.__version__}")
    
    # GPU 설정
    if tf.config.list_physical_devices('GPU'):
        print("GPU 사용 가능")
    else:
        print("CPU 모드로 실행")
    
    TENSORFLOW_AVAILABLE = True
except ImportError:
    print("TensorFlow를 사용할 수 없습니다. NumPy 구현을 사용합니다.")
    TENSORFLOW_AVAILABLE = False

# 시각화 설정
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
sns.set_palette("husl")

# 시드 설정
np.random.seed(42)
if TENSORFLOW_AVAILABLE:
    tf.random.set_seed(42)

print("환경 설정 완료!")

## 2. Universal Approximation Theorem 개념

### 2.1 정리의 정의

**Universal Approximation Theorem**는 다음과 같이 정의됩니다:

> 충분히 많은 수의 은닉 뉴런을 가진 단일 은닉층 피드포워드 신경망은 유한한 입력 공간에서 연속 함수를 임의의 정확도로 근사할 수 있다.

### 2.2 수학적 표현

컴팩트 집합 $K \subset \mathbb{R}^n$과 연속 함수 $f: K \rightarrow \mathbb{R}$, 그리고 $\epsilon > 0$에 대해, 
다음을 만족하는 단일 은닉층 신경망 $F$가 존재한다:

$$\sup_{x \in K} |f(x) - F(x)| < \epsilon$$

여기서 $F(x) = \sum_{i=1}^{N} w_i \sigma(v_i^T x + b_i)$이고, $\sigma$는 비상수 연속 시그모이드 함수입니다.

In [ ]:
# Universal Approximation Theorem 시각적 설명
def plot_uat_concept():
    """
    Universal Approximation Theorem의 개념을 시각적으로 설명
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    x = np.linspace(-3, 3, 1000)
    
    # 원본 함수들
    functions = [
        (lambda x: np.sin(2*x) + 0.5*np.cos(5*x), "복잡한 주기 함수"),
        (lambda x: np.exp(-x**2), "가우시안 함수"),
        (lambda x: np.where(x > 0, x**2, -x**2), "구간별 함수")
    ]
    
    # 각 함수에 대해 근사 실험
    for i, (func, title) in enumerate(functions):
        y_true = func(x)
        
        # 원본 함수 그래프
        axes[0, i].plot(x, y_true, 'b-', linewidth=2, label='원본 함수')
        axes[0, i].set_title(f'{title}')
        axes[0, i].grid(True, alpha=0.3)
        axes[0, i].legend()
        
        # 다양한 뉴런 수로 근사
        neuron_counts = [5, 20, 50]
        colors = ['red', 'green', 'orange']
        
        for j, (n_neurons, color) in enumerate(zip(neuron_counts, colors)):
            # 간단한 시그모이드 기반 근사 (개념적 시연)
            y_approx = approximate_function_concept(x, func, n_neurons)
            
            if j == 0:
                axes[1, i].plot(x, y_true, 'b-', linewidth=2, label='원본 함수')
            
            axes[1, i].plot(x, y_approx, '--', color=color, linewidth=1.5, 
                          label=f'{n_neurons}개 뉴런')
        
        axes[1, i].set_title(f'{title} - 근사 결과')
        axes[1, i].grid(True, alpha=0.3)
        axes[1, i].legend()
    
    plt.tight_layout()
    plt.show()

def approximate_function_concept(x, func, n_neurons):
    """
    개념적인 함수 근사 (실제 신경망 학습 없이 시연용)
    """
    y_true = func(x)
    
    # 무작위 시그모이드 함수들의 선형 결합으로 근사
    np.random.seed(42)
    y_approx = np.zeros_like(x)
    
    for i in range(n_neurons):
        # 무작위 파라미터
        w = np.random.randn() * 0.5
        b = np.random.randn() * 2
        a = np.random.randn() * 0.1
        
        # 시그모이드 함수
        sigmoid = 1 / (1 + np.exp(-(a * x + b)))
        y_approx += w * sigmoid
    
    # 스케일링으로 대략적인 매칭
    y_approx = y_approx * np.std(y_true) / np.std(y_approx)
    y_approx = y_approx + np.mean(y_true) - np.mean(y_approx)
    
    return y_approx

# 개념 시각화 실행
plot_uat_concept()

## 3. Universal Approximation Theorem 증명

### 3.1 증명의 핵심 아이디어

증명은 다음 단계로 구성됩니다:

1. **Step 1**: 시그모이드 함수로 계단 함수 근사
2. **Step 2**: 계단 함수로 연속 함수 근사
3. **Step 3**: 선형 결합으로 임의 정확도 달성

### 3.2 시그모이드 함수의 성질

In [ ]:
def demonstrate_proof_steps():
    """
    Universal Approximation Theorem 증명 과정을 시각적으로 보여줌
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    x = np.linspace(-5, 5, 1000)
    
    # Step 1: 시그모이드로 계단 함수 근사
    axes[0, 0].set_title('Step 1: 시그모이드 → 계단 함수')
    
    # 다양한 기울기의 시그모이드
    slopes = [1, 5, 20, 100]
    colors = ['blue', 'green', 'orange', 'red']
    
    for slope, color in zip(slopes, colors):
        sigmoid = 1 / (1 + np.exp(-slope * x))
        axes[0, 0].plot(x, sigmoid, color=color, label=f'기울기={slope}')
    
    # 이상적인 계단 함수
    step = np.where(x >= 0, 1, 0)
    axes[0, 0].plot(x, step, 'k--', linewidth=2, label='이상적 계단함수')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_xlabel('x')
    axes[0, 0].set_ylabel('σ(ax)')
    
    # Step 2: 계단 함수 조합으로 구간별 상수함수 만들기
    axes[0, 1].set_title('Step 2: 계단 함수 조합')
    
    # 구간 [1, 2]에서 높이 1인 함수 만들기
    steep = 50
    step1 = 1 / (1 + np.exp(-steep * (x - 1)))  # x=1에서 올라감
    step2 = 1 / (1 + np.exp(-steep * (x - 2)))  # x=2에서 올라감
    box = step1 - step2  # 구간 [1,2]에서만 1
    
    axes[0, 1].plot(x, step1, 'b--', alpha=0.7, label='σ(50(x-1))')
    axes[0, 1].plot(x, step2, 'r--', alpha=0.7, label='σ(50(x-2))')
    axes[0, 1].plot(x, box, 'g-', linewidth=3, label='차이 (구간함수)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_xlabel('x')
    axes[0, 1].set_ylabel('함수값')
    
    # Step 3: 구간별 상수함수들의 합으로 임의 함수 근사
    axes[0, 2].set_title('Step 3: 구간함수들의 합')
    
    # 목표 함수
    target_func = np.sin(x) * np.exp(-x**2/8)
    axes[0, 2].plot(x, target_func, 'b-', linewidth=2, label='목표 함수')
    
    # 구간별 근사
    n_intervals = 20
    x_intervals = np.linspace(-5, 5, n_intervals+1)
    approximation = np.zeros_like(x)
    
    for i in range(n_intervals):
        left, right = x_intervals[i], x_intervals[i+1]
        mid = (left + right) / 2
        height = np.sin(mid) * np.exp(-mid**2/8)  # 구간 중점에서의 함수값
        
        # 구간 함수 생성
        step_left = 1 / (1 + np.exp(-steep * (x - left)))
        step_right = 1 / (1 + np.exp(-steep * (x - right)))
        interval_func = height * (step_left - step_right)
        
        approximation += interval_func
    
    axes[0, 2].plot(x, approximation, 'r--', linewidth=2, label=f'{n_intervals}개 구간 근사')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    axes[0, 2].set_xlabel('x')
    axes[0, 2].set_ylabel('함수값')
    
    # 근사 오차 분석
    axes[1, 0].set_title('근사 오차 vs 뉴런 수')
    
    neuron_counts = [5, 10, 20, 50, 100, 200]
    errors = []
    
    for n_neurons in neuron_counts:
        # 간단한 오차 계산 (개념적)
        error = 1 / np.sqrt(n_neurons)  # 이론적으로 뉴런 수에 반비례
        errors.append(error)
    
    axes[1, 0].loglog(neuron_counts, errors, 'bo-', linewidth=2, markersize=8)
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_xlabel('뉴런 수')
    axes[1, 0].set_ylabel('근사 오차')
    
    # 다차원에서의 근사
    axes[1, 1].set_title('2차원 함수 근사')
    
    # 2D 가우시안 함수
    x2d = np.linspace(-2, 2, 50)
    y2d = np.linspace(-2, 2, 50)
    X2d, Y2d = np.meshgrid(x2d, y2d)
    Z = np.exp(-(X2d**2 + Y2d**2))
    
    im = axes[1, 1].contourf(X2d, Y2d, Z, levels=20, cmap='viridis')
    axes[1, 1].set_xlabel('x₁')
    axes[1, 1].set_ylabel('x₂')
    plt.colorbar(im, ax=axes[1, 1])
    
    # 활성화 함수 비교
    axes[1, 2].set_title('다양한 활성화 함수')
    
    x_act = np.linspace(-3, 3, 1000)
    
    # 시그모이드
    sigmoid = 1 / (1 + np.exp(-x_act))
    axes[1, 2].plot(x_act, sigmoid, label='Sigmoid', linewidth=2)
    
    # tanh
    tanh = np.tanh(x_act)
    axes[1, 2].plot(x_act, tanh, label='Tanh', linewidth=2)
    
    # ReLU
    relu = np.maximum(0, x_act)
    axes[1, 2].plot(x_act, relu, label='ReLU', linewidth=2)
    
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].set_xlabel('x')
    axes[1, 2].set_ylabel('σ(x)')
    
    plt.tight_layout()
    plt.show()

# 증명 과정 시각화
demonstrate_proof_steps()

## 4. 실제 신경망으로 Universal Approximation 실험

In [ ]:
if TENSORFLOW_AVAILABLE:
    def universal_approximation_experiment():
        """
        실제 신경망을 사용한 Universal Approximation 실험
        """
        print("=== Universal Approximation 실험 ===\n")
        
        # 테스트할 함수들 정의
        test_functions = {
            'sine_wave': lambda x: np.sin(2*np.pi*x),
            'gaussian': lambda x: np.exp(-5*(x-0.5)**2),
            'step': lambda x: np.where(x > 0.5, 1.0, 0.0),
            'polynomial': lambda x: 4*x**3 - 3*x**2 + 2*x - 1,
            'abs': lambda x: np.abs(x - 0.5)
        }
        
        # 입력 데이터 생성
        n_points = 1000
        x_train = np.random.uniform(0, 1, n_points)
        x_test = np.linspace(0, 1, 200)
        
        # 결과 저장
        results = {}
        
        # 각 함수에 대해 실험
        for func_name, func in test_functions.items():
            print(f"함수 '{func_name}' 근사 실험...")
            
            y_train = func(x_train)
            y_test_true = func(x_test)
            
            # 다양한 뉴런 수로 실험
            neuron_counts = [5, 10, 20, 50, 100]
            func_results = []
            
            for n_neurons in neuron_counts:
                # 모델 생성
                model = models.Sequential([
                    layers.Dense(n_neurons, activation='sigmoid', input_shape=(1,)),
                    layers.Dense(1, activation='linear')
                ])
                
                model.compile(
                    optimizer=optimizers.Adam(learning_rate=0.01),
                    loss='mse'
                )
                
                # 학습
                history = model.fit(
                    x_train, y_train,
                    epochs=1000,
                    batch_size=32,
                    verbose=0
                )
                
                # 예측 및 평가
                y_test_pred = model.predict(x_test, verbose=0).flatten()
                mse = np.mean((y_test_true - y_test_pred)**2)
                
                func_results.append({
                    'neurons': n_neurons,
                    'mse': mse,
                    'predictions': y_test_pred
                })
                
                print(f"  {n_neurons:3d} 뉴런: MSE = {mse:.6f}")
            
            results[func_name] = {
                'true_values': y_test_true,
                'results': func_results
            }
            print()
        
        return results, x_test
    
    def plot_approximation_results(results, x_test):
        """
        근사 결과 시각화
        """
        n_functions = len(results)
        fig, axes = plt.subplots(n_functions, 2, figsize=(15, 4*n_functions))
        
        if n_functions == 1:
            axes = axes.reshape(1, -1)
        
        for i, (func_name, data) in enumerate(results.items()):
            y_true = data['true_values']
            func_results = data['results']
            
            # 좌측: 근사 결과 비교
            axes[i, 0].plot(x_test, y_true, 'b-', linewidth=3, label='원본 함수')
            
            colors = ['red', 'green', 'orange', 'purple', 'brown']
            for j, result in enumerate(func_results):
                n_neurons = result['neurons']
                y_pred = result['predictions']
                
                axes[i, 0].plot(x_test, y_pred, '--', color=colors[j], 
                              linewidth=2, label=f'{n_neurons} 뉴런')
            
            axes[i, 0].set_title(f'{func_name} 함수 근사')
            axes[i, 0].set_xlabel('x')
            axes[i, 0].set_ylabel('f(x)')
            axes[i, 0].legend()
            axes[i, 0].grid(True, alpha=0.3)
            
            # 우측: MSE vs 뉴런 수
            neurons = [r['neurons'] for r in func_results]
            mses = [r['mse'] for r in func_results]
            
            axes[i, 1].loglog(neurons, mses, 'ro-', linewidth=2, markersize=8)
            axes[i, 1].set_title(f'{func_name} - MSE vs 뉴런 수')
            axes[i, 1].set_xlabel('뉴런 수')
            axes[i, 1].set_ylabel('Mean Squared Error')
            axes[i, 1].grid(True, alpha=0.3)
            
            # MSE 값들을 점 위에 표시
            for n, mse in zip(neurons, mses):
                axes[i, 1].annotate(f'{mse:.4f}', (n, mse), 
                                  textcoords="offset points", xytext=(0,10), ha='center')
        
        plt.tight_layout()
        plt.show()
    
    # 실험 실행
    results, x_test = universal_approximation_experiment()
    plot_approximation_results(results, x_test)
    
else:
    print("TensorFlow를 사용할 수 없어 신경망 실험을 건너뜁니다.")

## 5. Universal Approximation Theorem의 한계

### 5.1 이론과 실제의 차이

Universal Approximation Theorem은 **존재성(existence)**만을 보장하며, 다음과 같은 중요한 한계가 있습니다:

1. **학습 가능성을 보장하지 않음**: 이론적으로 근사 가능한 가중치가 존재하지만, 경사하강법으로 찾을 수 있다는 보장은 없음
2. **필요한 뉴런 수에 대한 정보 없음**: 주어진 정확도를 달성하기 위해 몇 개의 뉴런이 필요한지 알 수 없음
3. **일반화 능력과는 무관**: 훈련 데이터 근사와 일반화는 별개 문제
4. **실제 구현의 제약**: 부동소수점 정밀도, 메모리 제한 등

In [ ]:
def demonstrate_limitations():
    """
    Universal Approximation Theorem의 한계 시연
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. 차원의 저주
    axes[0, 0].set_title('차원의 저주')
    
    dimensions = [1, 2, 3, 4, 5]
    required_neurons = [10, 100, 1000, 10000, 100000]  # 가상의 필요 뉴런 수
    
    axes[0, 0].semilogy(dimensions, required_neurons, 'ro-', linewidth=2, markersize=8)
    axes[0, 0].set_xlabel('입력 차원')
    axes[0, 0].set_ylabel('필요 뉴런 수 (추정)')
    axes[0, 0].grid(True, alpha=0.3)
    
    for d, n in zip(dimensions, required_neurons):
        axes[0, 0].annotate(f'{n:,}', (d, n), textcoords="offset points", 
                          xytext=(0,10), ha='center')
    
    # 2. 학습의 어려움
    axes[0, 1].set_title('학습 수렴성 문제')
    
    # 시뮬레이션된 학습 곡선들
    epochs = np.arange(1, 501)
    
    # 좋은 경우
    loss_good = 1.0 * np.exp(-epochs/100) + 0.01
    axes[0, 1].plot(epochs, loss_good, 'g-', linewidth=2, label='수렴 성공')
    
    # 나쁜 경우 1: 지역 최솟값
    loss_local = 0.5 + 0.3 * np.exp(-epochs/50)
    axes[0, 1].plot(epochs, loss_local, 'r-', linewidth=2, label='지역 최솟값')
    
    # 나쁜 경우 2: 느린 수렴
    loss_slow = 1.0 / np.sqrt(epochs/10 + 1) + 0.1
    axes[0, 1].plot(epochs, loss_slow, 'orange', linewidth=2, label='느린 수렴')
    
    axes[0, 1].set_xlabel('에포크')
    axes[0, 1].set_ylabel('손실')
    axes[0, 1].set_yscale('log')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. 일반화 vs 암기
    axes[0, 2].set_title('과적합 문제')
    
    # 훈련 및 검증 손실
    train_loss = 1.0 * np.exp(-epochs/80)
    val_loss = 1.0 * np.exp(-epochs/80) + 0.2 * np.sin(epochs/50) * np.exp(-epochs/200)
    val_loss = np.maximum(val_loss, train_loss + 0.05)  # 검증 손실이 훈련보다 낮지 않도록
    
    axes[0, 2].plot(epochs, train_loss, 'b-', linewidth=2, label='훈련 손실')
    axes[0, 2].plot(epochs, val_loss, 'r-', linewidth=2, label='검증 손실')
    axes[0, 2].set_xlabel('에포크')
    axes[0, 2].set_ylabel('손실')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # 4. 활성화 함수의 영향
    axes[1, 0].set_title('활성화 함수별 근사 능력')
    
    activation_performance = {
        'Sigmoid': 0.85,
        'Tanh': 0.88,
        'ReLU': 0.92,
        'ELU': 0.90,
        'Swish': 0.93
    }
    
    activations = list(activation_performance.keys())
    performances = list(activation_performance.values())
    
    bars = axes[1, 0].bar(activations, performances, color=['blue', 'green', 'red', 'orange', 'purple'])
    axes[1, 0].set_ylabel('근사 정확도')
    axes[1, 0].set_ylim(0.8, 0.95)
    
    for bar, perf in zip(bars, performances):
        axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                       f'{perf:.2f}', ha='center', va='bottom')
    
    # 5. 메모리 및 계산 복잡도
    axes[1, 1].set_title('메모리 사용량 vs 뉴런 수')
    
    neurons = np.array([10, 50, 100, 500, 1000, 5000])
    # 메모리는 대략 O(n^2) (가중치 행렬)
    memory_mb = (neurons * 100 * 4) / (1024**2)  # 4바이트 float 가정
    
    axes[1, 1].loglog(neurons, memory_mb, 'bo-', linewidth=2, markersize=8)
    axes[1, 1].set_xlabel('뉴런 수')
    axes[1, 1].set_ylabel('메모리 사용량 (MB)')
    axes[1, 1].grid(True, alpha=0.3)
    
    # 6. 실제 적용에서의 고려사항
    axes[1, 2].set_title('실제 성능 제약 요소')
    axes[1, 2].axis('off')
    
    limitations_text = """
    Universal Approximation Theorem의 실제 한계:
    
    📊 이론적 한계:
    • 존재성만 보장, 학습 가능성 보장 안 함
    • 필요한 뉴런 수에 대한 정보 없음
    • 수렴 속도에 대한 보장 없음
    
    💻 실용적 한계:
    • 메모리 및 계산 자원 제약
    • 지역 최솟값 문제
    • 과적합 위험성
    • 차원의 저주
    
    🚀 극복 방안:
    • 적절한 정규화 기법 사용
    • 배치 정규화, 드롭아웃 등
    • 앙상블 방법
    • 전이 학습
    """
    
    axes[1, 2].text(0.05, 0.95, limitations_text, transform=axes[1, 2].transAxes,
                    fontsize=11, verticalalignment='top',
                    bbox=dict(boxstyle="round,pad=0.5", facecolor="lightblue", alpha=0.8))
    
    plt.tight_layout()
    plt.show()

# 한계 시연
demonstrate_limitations()

## 6. 다양한 활성화 함수에서의 Universal Approximation

In [ ]:
def compare_activation_universality():
    """
    다양한 활성화 함수의 universal approximation 능력 비교
    """
    # 활성화 함수들 정의
    x = np.linspace(-3, 3, 1000)
    
    activations = {
        'Sigmoid': 1 / (1 + np.exp(-x)),
        'Tanh': np.tanh(x),
        'ReLU': np.maximum(0, x),
        'Leaky ReLU': np.where(x > 0, x, 0.1 * x),
        'ELU': np.where(x > 0, x, np.exp(x) - 1),
        'Swish': x * (1 / (1 + np.exp(-x)))
    }
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.ravel()
    
    for i, (name, y) in enumerate(activations.items()):
        axes[i].plot(x, y, linewidth=3, color=f'C{i}')
        axes[i].set_title(f'{name} 활성화 함수')
        axes[i].grid(True, alpha=0.3)
        axes[i].set_xlabel('x')
        axes[i].set_ylabel(f'{name}(x)')
        
        # 각 활성화 함수의 특성 설명
        properties = {
            'Sigmoid': '• 범위: (0,1)\n• 미분 가능\n• 포화 문제',
            'Tanh': '• 범위: (-1,1)\n• 0 중심\n• 포화 문제',
            'ReLU': '• 범위: [0,∞)\n• 계산 효율적\n• 죽는 뉴런 문제',
            'Leaky ReLU': '• 음수 기울기 유지\n• 죽는 뉴런 방지',
            'ELU': '• 부드러운 곡선\n• 음수 출력 가능',
            'Swish': '• 자기 제어\n• 최근 각광받는 함수'
        }
        
        axes[i].text(0.02, 0.98, properties[name], transform=axes[i].transAxes,
                    fontsize=9, verticalalignment='top',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    # Universal Approximation에 필요한 조건 설명
    print("=== Universal Approximation을 위한 활성화 함수 조건 ===")
    print("")
    print("Cybenko (1989)의 정리:")
    print("• 시그모이드 형태의 연속 함수면 충분")
    print("• σ(x) = 1/(1+e^(-x)) 같은 S자 형태")
    print("")
    print("Hornik (1991)의 일반화:")
    print("• 비다항식이고 유계가 아닌 연속 함수면 충분")
    print("• ReLU도 포함됨")
    print("")
    print("현재 알려진 사실:")
    print("✓ Sigmoid, Tanh: 고전적인 증명")
    print("✓ ReLU: 더 강력한 근사 능력")
    print("✓ Leaky ReLU, ELU, Swish: 모두 가능")
    print("✗ Linear: 선형 조합의 한계")

# 활성화 함수 비교
compare_activation_universality()

## 7. 실습 정리 및 핵심 개념

### 7.1 Universal Approximation Theorem 요약

**핵심 메시지**: 단일 은닉층 신경망은 이론적으로 모든 연속 함수를 근사할 수 있다.

### 7.2 중요한 함의

1. **이론적 기초**: 신경망의 표현력에 대한 수학적 보장
2. **깊이 vs 너비**: 깊은 네트워크가 더 효율적일 수 있음
3. **실용적 고려사항**: 학습 가능성과 일반화는 별개 문제

### 7.3 실무 적용 가이드라인

- **네트워크 설계**: 충분한 용량 확보, 하지만 과적합 주의
- **활성화 함수**: ReLU 계열이 일반적으로 좋은 성능
- **정규화**: 이론적 근사 능력과 실제 일반화 성능의 균형
- **최적화**: 지역 최솟값 문제 고려한 초기화와 학습률 설정

In [ ]:
# 실습 요약
print("="*80)
print("          Universal Approximation Theorem 실습 완료")
print("="*80)
print()
print("📚 학습한 주요 내용:")
print("1. Universal Approximation Theorem의 정의와 의미")
print("2. 시그모이드 함수를 이용한 증명 과정")
print("3. 실제 신경망을 통한 함수 근사 실험")
print("4. 이론과 실제의 차이점 및 한계")
print("5. 다양한 활성화 함수의 근사 능력")
print()
print("🔑 핵심 포인트:")
print("• 존재성 ≠ 학습 가능성")
print("• 근사 ≠ 일반화")
print("• 이론적 보장 ≠ 실용적 효율성")
print()
print("🚀 다음 단계:")
print("• 깊은 네트워크의 표현력 이론")
print("• 최적화 이론과 경사하강법")
print("• 일반화 이론과 PAC 학습")
print("="*80)

## 추가 연습 문제

### 연습 문제 1: 다차원 함수 근사
2차원 입력을 받는 복잡한 함수를 신경망으로 근사해보세요.

### 연습 문제 2: 불연속 함수 근사
계단 함수나 톱니파 같은 불연속 함수의 근사 성능을 분석해보세요.

### 연습 문제 3: 활성화 함수 설계
새로운 활성화 함수를 설계하고 Universal Approximation 능력을 테스트해보세요.

### 연습 문제 4: 노이즈가 있는 데이터
노이즈가 있는 함수 데이터에서 근사 성능과 일반화 성능을 비교해보세요.